In [1]:
# =========================================
# 🏨 HOTEL LOCATION SCORING (Chennai)
# =========================================

import pandas as pd
import numpy as np
from geopy.distance import geodesic
import folium

# ======================
# 1️⃣ PARAMETERS (FIXED)
# ======================
n = 1     # grid size in km
m = 2     # scan radius in km

# More accurate Chennai bounding box
min_lat, max_lat = 12.85, 13.20
min_lon, max_lon = 80.13, 80.29

# ======================
# 2️⃣ ACCURATE GRID CREATION
# ======================
def create_grid(min_lat, max_lat, min_lon, max_lon, cell_size_km):
    lat_deg_per_km = 1 / 110.574
    lon_deg_per_km = 1 / 111.320
    
    lat_step = cell_size_km * lat_deg_per_km
    lon_step = cell_size_km * lon_deg_per_km
    
    cells = []
    cell_id = 0
    
    lat_cells = int((max_lat - min_lat) / lat_step)
    lon_cells = int((max_lon - min_lon) / lon_step)
    
    print(f"Grid dimensions: {lat_cells} × {lon_cells} = {lat_cells * lon_cells} total cells")
    
    for i in range(lat_cells):
        for j in range(lon_cells):
            cell_id += 1
            center_lat = min_lat + (i * lat_step) + (lat_step / 2)
            center_lon = min_lon + (j * lon_step) + (lon_step / 2)
            
            cells.append({
                "CellID": cell_id,
                "Lat": center_lat,
                "Lon": center_lon
            })
    
    return pd.DataFrame(cells)

# ======================
# 3️⃣ WEIGHT CONFIGURATION
# ======================
weights = {
    "tourist_places": 3.5,     # tourist demand
    "offices": 2.5,            # business travelers
    "schools": -2.0,           # noise/traffic
    "hardware_shops": -1.5,    # unattractive zone
    "hotels": -3.0             # booking competition
}

# ======================
# 4️⃣ LOAD CSV FILES
# ======================
poi_files = {
    "tourist_places": "chennai_tourist_places.csv",
    "offices": "chennai_offices.csv",
    "schools": "chennai_schools.csv",
    "hardware_shops": "chennai_hardware_shops.csv",
    "hotels": "chennai_hotels.csv"
}

pois = {}
for key, file in poi_files.items():
    try:
        df = pd.read_csv(file)
        df = df.rename(columns={'lat': 'Latitude', 'lon': 'Longitude'})
        pois[key] = df[['Latitude', 'Longitude']]
        print(f"✅ Loaded {len(df)} {key}")
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")
        pois[key] = pd.DataFrame(columns=['Latitude', 'Longitude'])

# ======================
# 5️⃣ CREATE GRID
# ======================
df_cells = create_grid(min_lat, max_lat, min_lon, max_lon, n)
print(f"✅ Created grid with {len(df_cells)} cells")

# ======================
# 6️⃣ GAUSSIAN DISTANCE DECAY
# ======================
def gaussian_weight(distance_km, sigma):
    return np.exp(- (distance_km ** 2) / (2 * sigma ** 2))

# ======================
# 7️⃣ CALCULATE SCORES
# ======================
sigma = m / 2
results = []

print("Calculating scores for each cell...")
for idx, cell in df_cells.iterrows():
    if idx % 50 == 0:
        print(f"Processing cell {idx+1}/{len(df_cells)}")
    
    center = (cell.Lat, cell.Lon)
    score_components = {}
    total_raw_score = 0

    for key, df in pois.items():
        influence = 0
        for _, row in df.iterrows():
            try:
                dist = geodesic(center, (row["Latitude"], row["Longitude"])).km
                if dist <= m:
                    influence += gaussian_weight(dist, sigma)
            except:
                continue
        
        weighted_score = influence * weights[key]
        score_components[key] = weighted_score
        total_raw_score += weighted_score

    results.append({
        "CellID": cell.CellID,
        "Lat": cell.Lat,
        "Lon": cell.Lon,
        **score_components,
        "RawScore": total_raw_score
    })

df_results = pd.DataFrame(results)

# ======================
# 8️⃣ NORMALIZE TO 0–1000
# ======================
min_s, max_s = df_results["RawScore"].min(), df_results["RawScore"].max()
if max_s > min_s:
    df_results["Score_0_1000"] = ((df_results["RawScore"] - min_s) / (max_s - min_s)) * 1000
else:
    df_results["Score_0_1000"] = 500

df_results["Rank"] = df_results["Score_0_1000"].rank(ascending=False).astype(int)

# ======================
# 9️⃣ SAVE OUTPUT
# ======================
df_results.to_csv("hotel_location_scores.csv", index=False)
print(f"✅ Hotel location scores saved with {len(df_results)} cells")

# Show top locations
print("\n🏆 TOP 10 HOTEL LOCATIONS:")
top_10 = df_results.nlargest(10, 'Score_0_1000')[['CellID', 'Lat', 'Lon', 'Score_0_1000', 'Rank']]
print(top_10.to_string(index=False))

✅ Loaded 1109 tourist_places
✅ Loaded 410 offices
✅ Loaded 209 schools
✅ Loaded 269 hardware_shops
✅ Loaded 297 hotels
Grid dimensions: 38 × 17 = 646 total cells
✅ Created grid with 646 cells
Calculating scores for each cell...
Processing cell 1/646
Processing cell 51/646
Processing cell 101/646
Processing cell 151/646
Processing cell 201/646
Processing cell 251/646
Processing cell 301/646
Processing cell 351/646
Processing cell 401/646
Processing cell 451/646
Processing cell 501/646
Processing cell 551/646
Processing cell 601/646
✅ Hotel location scores saved with 646 cells

🏆 TOP 10 HOTEL LOCATIONS:
 CellID       Lat       Lon  Score_0_1000  Rank
  334.0 13.026352 80.224323   1000.000000     1
  317.0 13.017309 80.224323    967.532535     2
  316.0 13.017309 80.215340    952.566838     3
  333.0 13.026352 80.215340    944.210727     4
  315.0 13.017309 80.206356    838.000659     5
  318.0 13.017309 80.233306    831.701078     6
  335.0 13.026352 80.233306    802.748660     7
  351.0

In [2]:
# ======================
# 🔟 MAP VISUALIZATION
# ======================
mymap = folium.Map(location=[13.0827, 80.2707], zoom_start=11)

for _, row in df_results.iterrows():
    if row.Score_0_1000 > df_results['Score_0_1000'].quantile(0.9):
        color = "purple"
    elif row.Score_0_1000 > df_results['Score_0_1000'].quantile(0.5):
        color = "orange"
    else:
        color = "red"
    
    folium.CircleMarker(
        location=[row.Lat, row.Lon],
        radius=5,
        color=color,
        fill=True,
        popup=(f"🏨 Cell {row.CellID}<br>"
               f"Score: {row.Score_0_1000:.1f}<br>"
               f"Rank: {row.Rank}")
    ).add_to(mymap)

mymap.save("hotel_heatmap.html")
print("🗺️ Heatmap saved as hotel_heatmap.html")

🗺️ Heatmap saved as hotel_heatmap.html


In [3]:
# =========================================
# 👕 CLOTHING SHOP LOCATION SCORING (Chennai)
# =========================================

import pandas as pd
import numpy as np
from geopy.distance import geodesic
import folium

# ======================
# 1️⃣ PARAMETERS (FIXED)
# ======================
n = 1     # grid size in km
m = 2     # scan radius in km

# More accurate Chennai bounding box
min_lat, max_lat = 12.85, 13.20
min_lon, max_lon = 80.13, 80.29

# ======================
# 2️⃣ ACCURATE GRID CREATION
# ======================
def create_grid(min_lat, max_lat, min_lon, max_lon, cell_size_km):
    lat_deg_per_km = 1 / 110.574
    lon_deg_per_km = 1 / 111.320
    
    lat_step = cell_size_km * lat_deg_per_km
    lon_step = cell_size_km * lon_deg_per_km
    
    cells = []
    cell_id = 0
    
    lat_cells = int((max_lat - min_lat) / lat_step)
    lon_cells = int((max_lon - min_lon) / lon_step)
    
    print(f"Grid dimensions: {lat_cells} × {lon_cells} = {lat_cells * lon_cells} total cells")
    
    for i in range(lat_cells):
        for j in range(lon_cells):
            cell_id += 1
            center_lat = min_lat + (i * lat_step) + (lat_step / 2)
            center_lon = min_lon + (j * lon_step) + (lon_step / 2)
            
            cells.append({
                "CellID": cell_id,
                "Lat": center_lat,
                "Lon": center_lon
            })
    
    return pd.DataFrame(cells)

# ======================
# 3️⃣ WEIGHT CONFIGURATION
# ======================
weights = {
    "apartments": 3.0,        # family shoppers
    "tourist_places": 2.5,    # souvenir demand
    "supermarkets": 1.5,      # complementary zone
    "clothing_shops": -3.0    # high competition
}

# ======================
# 4️⃣ LOAD CSV FILES
# ======================
poi_files = {
    "apartments": "chennai_apartments.csv",
    "tourist_places": "chennai_tourist_places.csv",
    "supermarkets": "chennai_supermarkets.csv",
    "clothing_shops": "chennai_clothing_shops.csv"
}

pois = {}
for key, file in poi_files.items():
    try:
        df = pd.read_csv(file)
        df = df.rename(columns={'lat': 'Latitude', 'lon': 'Longitude'})
        pois[key] = df[['Latitude', 'Longitude']]
        print(f"✅ Loaded {len(df)} {key}")
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")
        pois[key] = pd.DataFrame(columns=['Latitude', 'Longitude'])

# ======================
# 5️⃣ CREATE GRID
# ======================
df_cells = create_grid(min_lat, max_lat, min_lon, max_lon, n)
print(f"✅ Created grid with {len(df_cells)} cells")

# ======================
# 6️⃣ GAUSSIAN DISTANCE DECAY
# ======================
def gaussian_weight(distance_km, sigma):
    return np.exp(- (distance_km ** 2) / (2 * sigma ** 2))

# ======================
# 7️⃣ CALCULATE SCORES
# ======================
sigma = m / 2
results = []

print("Calculating scores for each cell...")
for idx, cell in df_cells.iterrows():
    if idx % 50 == 0:
        print(f"Processing cell {idx+1}/{len(df_cells)}")
    
    center = (cell.Lat, cell.Lon)
    score_components = {}
    total_raw_score = 0

    for key, df in pois.items():
        influence = 0
        for _, row in df.iterrows():
            try:
                dist = geodesic(center, (row["Latitude"], row["Longitude"])).km
                if dist <= m:
                    influence += gaussian_weight(dist, sigma)
            except:
                continue
        
        weighted_score = influence * weights[key]
        score_components[key] = weighted_score
        total_raw_score += weighted_score

    results.append({
        "CellID": cell.CellID,
        "Lat": cell.Lat,
        "Lon": cell.Lon,
        **score_components,
        "RawScore": total_raw_score
    })

df_results = pd.DataFrame(results)

# ======================
# 8️⃣ NORMALIZE TO 0–1000
# ======================
min_s, max_s = df_results["RawScore"].min(), df_results["RawScore"].max()
if max_s > min_s:
    df_results["Score_0_1000"] = ((df_results["RawScore"] - min_s) / (max_s - min_s)) * 1000
else:
    df_results["Score_0_1000"] = 500

df_results["Rank"] = df_results["Score_0_1000"].rank(ascending=False).astype(int)

# ======================
# 9️⃣ SAVE OUTPUT
# ======================
df_results.to_csv("clothing_shop_location_scores.csv", index=False)
print(f"✅ Clothing shop location scores saved with {len(df_results)} cells")

# Show top locations
print("\n🏆 TOP 10 CLOTHING SHOP LOCATIONS:")
top_10 = df_results.nlargest(10, 'Score_0_1000')[['CellID', 'Lat', 'Lon', 'Score_0_1000', 'Rank']]
print(top_10.to_string(index=False))

# ======================
# 🔟 MAP VISUALIZATION
# ======================
mymap = folium.Map(location=[13.0827, 80.2707], zoom_start=11)

for _, row in df_results.iterrows():
    if row.Score_0_1000 > df_results['Score_0_1000'].quantile(0.85):
        color = "green"
    elif row.Score_0_1000 > df_results['Score_0_1000'].quantile(0.5):
        color = "orange"
    else:
        color = "red"
    
    folium.CircleMarker(
        location=[row.Lat, row.Lon],
        radius=5,
        color=color,
        fill=True,
        popup=(f"👕 Cell {row.CellID}<br>"
               f"Score: {row.Score_0_1000:.1f}<br>"
               f"Rank: {row.Rank}")
    ).add_to(mymap)

mymap.save("clothing_shop_heatmap.html")
print("🗺️ Heatmap saved as clothing_shop_heatmap.html")

✅ Loaded 3087 apartments
✅ Loaded 1109 tourist_places
✅ Loaded 218 supermarkets
✅ Loaded 778 clothing_shops
Grid dimensions: 38 × 17 = 646 total cells
✅ Created grid with 646 cells
Calculating scores for each cell...
Processing cell 1/646
Processing cell 51/646
Processing cell 101/646
Processing cell 151/646
Processing cell 201/646
Processing cell 251/646
Processing cell 301/646
Processing cell 351/646
Processing cell 401/646
Processing cell 451/646
Processing cell 501/646
Processing cell 551/646
Processing cell 601/646
✅ Clothing shop location scores saved with 646 cells

🏆 TOP 10 CLOTHING SHOP LOCATIONS:
 CellID       Lat       Lon  Score_0_1000  Rank
   60.0 12.881653 80.206356   1000.000000     1
  270.0 12.990178 80.260255    939.900756     2
   77.0 12.890697 80.206356    900.772386     3
  317.0 13.017309 80.224323    876.962395     4
  253.0 12.981134 80.260255    871.643465     5
  334.0 13.026352 80.224323    855.027664     6
  271.0 12.990178 80.269238    842.919336     7
  

In [4]:
# =========================================
# 🔧 HARDWARE SHOP LOCATION SCORING (Chennai)
# =========================================

import pandas as pd
import numpy as np
from geopy.distance import geodesic
import folium

# ======================
# 1️⃣ PARAMETERS (FIXED)
# ======================
n = 1     # grid size in km
m = 2     # scan radius in km

# More accurate Chennai bounding box
min_lat, max_lat = 12.85, 13.20
min_lon, max_lon = 80.13, 80.29

# ======================
# 2️⃣ ACCURATE GRID CREATION
# ======================
def create_grid(min_lat, max_lat, min_lon, max_lon, cell_size_km):
    lat_deg_per_km = 1 / 110.574
    lon_deg_per_km = 1 / 111.320
    
    lat_step = cell_size_km * lat_deg_per_km
    lon_step = cell_size_km * lon_deg_per_km
    
    cells = []
    cell_id = 0
    
    lat_cells = int((max_lat - min_lat) / lat_step)
    lon_cells = int((max_lon - min_lon) / lon_step)
    
    print(f"Grid dimensions: {lat_cells} × {lon_cells} = {lat_cells * lon_cells} total cells")
    
    for i in range(lat_cells):
        for j in range(lon_cells):
            cell_id += 1
            center_lat = min_lat + (i * lat_step) + (lat_step / 2)
            center_lon = min_lon + (j * lon_step) + (lon_step / 2)
            
            cells.append({
                "CellID": cell_id,
                "Lat": center_lat,
                "Lon": center_lon
            })
    
    return pd.DataFrame(cells)

# ======================
# 3️⃣ WEIGHT CONFIGURATION
# ======================
weights = {
    "apartments": 3.0,       # repair needs
    "offices": 2.5,          # maintenance contracts
    "schools": 1.5,          # infra maintenance
    "hardware_shops": -3.0   # reduced profit margin
}

# ======================
# 4️⃣ LOAD CSV FILES
# ======================
poi_files = {
    "apartments": "chennai_apartments.csv",
    "offices": "chennai_offices.csv",
    "schools": "chennai_schools.csv",
    "hardware_shops": "chennai_hardware_shops.csv"
}

pois = {}
for key, file in poi_files.items():
    try:
        df = pd.read_csv(file)
        df = df.rename(columns={'lat': 'Latitude', 'lon': 'Longitude'})
        pois[key] = df[['Latitude', 'Longitude']]
        print(f"✅ Loaded {len(df)} {key}")
    except Exception as e:
        print(f"❌ Error loading {file}: {e}")
        pois[key] = pd.DataFrame(columns=['Latitude', 'Longitude'])

# ======================
# 5️⃣ CREATE GRID
# ======================
df_cells = create_grid(min_lat, max_lat, min_lon, max_lon, n)
print(f"✅ Created grid with {len(df_cells)} cells")

# ======================
# 6️⃣ GAUSSIAN DISTANCE DECAY
# ======================
def gaussian_weight(distance_km, sigma):
    return np.exp(- (distance_km ** 2) / (2 * sigma ** 2))

# ======================
# 7️⃣ CALCULATE SCORES
# ======================
sigma = m / 2
results = []

print("Calculating scores for each cell...")
for idx, cell in df_cells.iterrows():
    if idx % 50 == 0:
        print(f"Processing cell {idx+1}/{len(df_cells)}")
    
    center = (cell.Lat, cell.Lon)
    score_components = {}
    total_raw_score = 0

    for key, df in pois.items():
        influence = 0
        for _, row in df.iterrows():
            try:
                dist = geodesic(center, (row["Latitude"], row["Longitude"])).km
                if dist <= m:
                    influence += gaussian_weight(dist, sigma)
            except:
                continue
        
        weighted_score = influence * weights[key]
        score_components[key] = weighted_score
        total_raw_score += weighted_score

    results.append({
        "CellID": cell.CellID,
        "Lat": cell.Lat,
        "Lon": cell.Lon,
        **score_components,
        "RawScore": total_raw_score
    })

df_results = pd.DataFrame(results)

# ======================
# 8️⃣ NORMALIZE TO 0–1000
# ======================
min_s, max_s = df_results["RawScore"].min(), df_results["RawScore"].max()
if max_s > min_s:
    df_results["Score_0_1000"] = ((df_results["RawScore"] - min_s) / (max_s - min_s)) * 1000
else:
    df_results["Score_0_1000"] = 500

df_results["Rank"] = df_results["Score_0_1000"].rank(ascending=False).astype(int)

# ======================
# 9️⃣ SAVE OUTPUT
# ======================
df_results.to_csv("hardware_shop_location_scores.csv", index=False)
print(f"✅ Hardware shop location scores saved with {len(df_results)} cells")

# Show top locations
print("\n🏆 TOP 10 HARDWARE SHOP LOCATIONS:")
top_10 = df_results.nlargest(10, 'Score_0_1000')[['CellID', 'Lat', 'Lon', 'Score_0_1000', 'Rank']]
print(top_10.to_string(index=False))

# ======================
# 🔟 MAP VISUALIZATION
# ======================
mymap = folium.Map(location=[13.0827, 80.2707], zoom_start=11)

for _, row in df_results.iterrows():
    if row.Score_0_1000 > df_results['Score_0_1000'].quantile(0.8):
        color = "blue"
    elif row.Score_0_1000 > df_results['Score_0_1000'].quantile(0.5):
        color = "orange"
    else:
        color = "red"
    
    folium.CircleMarker(
        location=[row.Lat, row.Lon],
        radius=5,
        color=color,
        fill=True,
        popup=(f"🔧 Cell {row.CellID}<br>"
               f"Score: {row.Score_0_1000:.1f}<br>"
               f"Rank: {row.Rank}")
    ).add_to(mymap)

mymap.save("hardware_shop_heatmap.html")
print("🗺️ Heatmap saved as hardware_shop_heatmap.html")

✅ Loaded 3087 apartments
✅ Loaded 410 offices
✅ Loaded 209 schools
✅ Loaded 269 hardware_shops
Grid dimensions: 38 × 17 = 646 total cells
✅ Created grid with 646 cells
Calculating scores for each cell...
Processing cell 1/646
Processing cell 51/646
Processing cell 101/646
Processing cell 151/646
Processing cell 201/646
Processing cell 251/646
Processing cell 301/646
Processing cell 351/646
Processing cell 401/646
Processing cell 451/646
Processing cell 501/646
Processing cell 551/646
Processing cell 601/646
✅ Hardware shop location scores saved with 646 cells

🏆 TOP 10 HARDWARE SHOP LOCATIONS:
 CellID       Lat       Lon  Score_0_1000  Rank
   60.0 12.881653 80.206356   1000.000000     1
  270.0 12.990178 80.260255    944.747221     2
   77.0 12.890697 80.206356    875.378897     3
  253.0 12.981134 80.260255    846.628211     4
  317.0 13.017309 80.224323    828.721020     5
  271.0 12.990178 80.269238    783.769700     6
  334.0 13.026352 80.224323    761.583927     7
   61.0 12.8816